In [ ]:
import os
import sys

os.chdir("..")  # Go up one directory
sys.path.append("src/")

import datetime
import gc
import time
from collections import defaultdict
from pprint import pprint

import einops
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display
from tqdm import tqdm

import wandb
from config import Config
from model import CASunGroup
from viz import (
    capture_snapshot,
    create_video,
    generate_nca_colors,
    get_shannon_entropy,
)
from world import World

In [ ]:
def store_video(raw_frames, name="output", pixel_size=2):
    frames = []
    for raw_frame in raw_frames:
        frames.append(capture_snapshot(raw_frame, nca_colors))

    upscaled_frames = []
    for frame in frames:
        # Upscale each pixel
        upscaled_frame = einops.repeat(
            frame,
            "c h w -> c (h repeat_h) (w repeat_w)",
            repeat_h=pixel_size,
            repeat_w=pixel_size,
        )
        upscaled_frames.append(upscaled_frame)

    del frames
    gc.collect()

    video_frames = einops.rearrange(
        (torch.stack(upscaled_frames) * 255).to(torch.uint8), "n c h w -> n h w c"
    )  # [N,C,H,W]
    create_video(video_frames, output_path=f"{name}.mp4", fps=30)

In [ ]:
def extract_nca_territory(
    grid: torch.Tensor, nca_idx: int, config: Config
) -> torch.Tensor:
    """
    Extract state channels for a specific NCA's territory.

    Args:
        grid: Full grid [C, H, W]
        nca_idx: Index of the NCA (0-indexed)
        config: Configuration object

    Returns:
        State channels masked to this NCA's territory [cell_state_dim, H, W]
    """
    # Get aliveness mask
    aliveness = grid[nca_idx + 1]  # +1 because sun is at index 0
    mask = aliveness > 0.1

    # Extract state channels
    start_idx = config.n_ncas + 1
    territory = grid[start_idx:].clone()

    territory = territory[:, mask]

    return territory


def track_stats(steps_taken, data, grid, group, config, stats=None, save_grid=False):
    """
    Track grid snapshot and compute per-NCA statistics.

    Args:
        steps_taken: Number of steps taken in the sim
        data: Dictionary to store results
        grid: Current grid state [B, C, H, W]
        group: CASunGroup model
        config: Configuration
        stats: Optional forward pass stats
    """

    # Only calculate stats for the first grid
    grid = grid[0].detach()

    nca_stats = {}

    for nca_i in range(config.n_ncas):
        # Population metrics
        territory = extract_nca_territory(grid, nca_i, config)
        alive_cells = territory.shape[1]

        # Information metrics (only if NCA has territory)
        if alive_cells > 0:
            # compression = get_compression_ratios(territory, img_mode=True).mean()
            entropy = get_shannon_entropy(territory).mean()
            # hoe = higher_order_entropy(territory, img_mode=True).mean()
        else:
            compression = 0.0
            entropy = 0.0
            hoe = 0.0

        nca_stats[f"nca_{nca_i}"] = {
            "alive_cells": alive_cells,
            # "compression_ratio": float(compression),
            "shannon_entropy": float(entropy),
            # "higher_order_entropy": float(hoe),
            "grad_norm": stats["grad_norms"][nca_i] if stats else 0.0,
        }

    # data["global_stats"].append(stats["forward"] if stats else None)
    data["nca_stats"].append(nca_stats)
    data["timesteps"].append(steps_taken)

    # Store grid snapshot (first batch element)
    if save_grid:
        data["grids"].append(grid.clone().cpu())

In [ ]:
# ------------ SETTINGS ------------

SETTINGS = {
    "load_method": "file",  # Either file / wandb
    "total_steps": 1_000_000,
    "plot_every": 5,
}

# ----------------------------------

assert SETTINGS["load_method"] == "wandb" or SETTINGS["load_method"] == "file"

In [ ]:
config = None

if SETTINGS["load_method"] == "file":
    # ---------- FILE PATH TO EDIT ------------
    file_path = "configs/tiny-config.json"
    # file_path = "2025-10-30_15-28-46/config.json"
    # -----------------------------------------

    config = Config.from_file(file_path)
elif SETTINGS["load_method"] == "wandb":
    # ---------- WANDB THINGS TO EDIT ------------
    entity = "[USERNAME]"
    project = "[PROJECT]"
    run_id = "[RUN_ID]"
    # --------------------------------------------

    api = wandb.Api()
    run = api.run(f"{entity}/{project}/{run_id}")

    run_config = run.config

    config = Config()

    for k, v in run_config.items():
        if "-" in k:
            continue
        setattr(config, k, v)
else:
    raise Exception("Invalid loading method")

In [ ]:
# ------------- MANUAL EDITS HERE -------------

# config.cell_state_dim = 8
# config.cell_hidden_dim = 16
# config.grid_size = [128, 128]
config.grid_size = [128, 128]
# config.n_hidden_layers = 10
# config.hidden_dim = 8
# config.n_ncas = 4
config.n_seeds = 4
# config.steps_per_update = 1
config.batch_size = 1
config.pool_size = 1
# config.softmax_temp = 0.9

config.seed = 13

# ---------------------------------------------

pprint(config)

In [ ]:
# Step 3: Run
nca_colors = generate_nca_colors(config.n_ncas)

# Solidify and then run
group = CASunGroup(config)
world = World(config)

grid = world.get_seed()

steps_taken = 0
vids_saved = 0
steps_to_add = config.steps_before_update + config.steps_per_update

timestep_data = {
    "grids": [],  # Full grid states
    "nca_stats": [],  # Per-NCA statistics (population + information)
    "global_stats": [],  # System-wide metrics
    "timesteps": [],  # Array of timesteps where you tracked
}

# Generate UUID + save config
experiment_id = datetime.datetime.fromtimestamp(time.time()).strftime(
    "%Y-%m-%d %H:%M:%S"
)
# experiment_id = uuid4()
experiment_dir = f"logs/{experiment_id}"
os.mkdir(experiment_dir)

config.save(f"{experiment_dir}/config.json")

pbar = tqdm(total=SETTINGS["total_steps"])

print(f"> starting experiment {experiment_id}")


In [ ]:
# Grid dimensions
aspect_ratio = config.grid_size[0] / config.grid_size[1]  # 16:1

# Set figure width and calculate height
fig_width = 12  # inches
fig_height = fig_width / aspect_ratio  # 1 inch

dpi_display = 100
dpi_save = 300

In [ ]:
while steps_taken < SETTINGS["total_steps"]:
    if steps_taken % SETTINGS["plot_every"] == 0:
        clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=dpi_display)
        ax.imshow(capture_snapshot(grid, nca_colors).permute(1, 2, 0))
        ax.set_title(f"Step: {steps_taken}")
        ax.axis("off")
        plt.tight_layout()

        plt.savefig(
            f"{experiment_dir}/{steps_taken:08d}", dpi=dpi_save, bbox_inches="tight"
        )

        # Display locally in the notebook
        display(fig)
        plt.close(fig)

        display(pbar)

    stats, grid, grids = world.step(group, grid)

    for i in range(len(grids)):
        track_stats(
            steps_taken + i, timestep_data, grids[i], group, config, save_grid=True
        )

    steps_taken += steps_to_add
    pbar.update(steps_to_add)

pbar.close()

## Analysis

Plot all the things here now.

Things that I want to tracK:

- pop. over time
- information per NCA over time
- with one NCA, does information decrease?
- can asal fm actually measure something meaningful?


In [ ]:
def show_nca_colors(colors):
    """Display a minimal color legend for NCAs."""
    fig, ax = plt.subplots(figsize=(12, 0.4), dpi=dpi_display)
    for i, color in enumerate(colors):
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, facecolor=color, edgecolor="none"))
    ax.set_xlim(0, len(colors))
    ax.set_ylim(0, 1)
    ax.set_xticks([i + 0.5 for i in range(len(colors))])
    ax.set_xticklabels([f"NCA {i}" for i in range(len(colors))], fontsize=8)
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    # plt.tight_layout()
    plt.show()


show_nca_colors(nca_colors)

In [ ]:
xs_idxs = np.arange(len(timestep_data["nca_stats"]))
xs = timestep_data["timesteps"]

fig, ax = plt.subplots(figsize=(12, 6))

ax.set_xlabel("Timestep")
ax.set_ylabel("Population")

for nca_i in range(config.n_ncas):
    territories = []

    for step in range(len(timestep_data["nca_stats"])):
        stats = timestep_data["nca_stats"][step]
        if f"nca_{nca_i}" in stats:
            territories.append(stats[f"nca_{nca_i}"]["alive_cells"])
        else:
            territories.append(0)

    color = nca_colors[nca_i][:3] if nca_i < len(nca_colors) else plt.cm.tab10(nca_i)

    ax.plot(xs_idxs, territories, color=color, linewidth=1, alpha=0.8)

plt.title("Territory size")
plt.tight_layout()

n_ticks = 10  # Number of ticks you want to show
tick_indices = np.linspace(0, len(xs_idxs) - 1, n_ticks, dtype=int)
ax.set_xticks(tick_indices)
ax.set_xticklabels([xs[i] for i in tick_indices])

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.savefig(f"{experiment_dir}/territory_{steps_taken}", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
xs_idxs = np.arange(len(timestep_data["nca_stats"]))
xs = timestep_data["timesteps"]

fig, ax = plt.subplots(figsize=(12, 6))

ax.set_xlabel("Timestep")
ax.set_ylabel("Avg. Bits Encoded/Channel")

for nca_i in range(config.n_ncas):
    ents = []

    for step in range(len(timestep_data["nca_stats"])):
        stats = timestep_data["nca_stats"][step]
        if f"nca_{nca_i}" in stats:
            ents.append(stats[f"nca_{nca_i}"]["shannon_entropy"])
        else:
            ents.append(0)

    color = nca_colors[nca_i][:3] if nca_i < len(nca_colors) else plt.cm.tab10(nca_i)

    ax.plot(xs_idxs, ents, "--", color=color, linewidth=1, alpha=0.8)

plt.title("Information encoded")
plt.tight_layout()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

n_ticks = 10  # Number of ticks you want to show
tick_indices = np.linspace(0, len(xs_idxs) - 1, n_ticks, dtype=int)
ax.set_xticks(tick_indices)
ax.set_xticklabels([xs[i] for i in tick_indices])

plt.savefig(f"{experiment_dir}/ent_{steps_taken}", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
## Q: Is there any relationship between the bits encoded & the population?

corrs = []

for nca_i in range(config.n_ncas):
    territories = []
    ents = []

    for step in range(len(timestep_data["nca_stats"])):
        stats = timestep_data["nca_stats"][step]
        if f"nca_{nca_i}" in stats:
            territories.append(stats[f"nca_{nca_i}"]["alive_cells"])
            ents.append(stats[f"nca_{nca_i}"]["shannon_entropy"])
        else:
            territories.append(0)
            ents.append(0)

    territories = np.array(territories)
    ents = np.array(ents)

    t_delta = territories[1:] - territories[:-1]
    e_delta = ents[1:] - ents[:-1]
    corr = np.corrcoef(territories, ents)[0, 1]
    corr_delta = np.corrcoef(t_delta, e_delta)[0, 1]
    corrs.append(corr)
    print(
        f"NCA {nca_i} base corr. {corr * 100:.2f}% | delta corr. {corr_delta * 100:.2f}%"
    )

In [ ]:
## Q: Less # of NCAs = less bits encoded?

n_alive = [
    sum(1 for v in ts.values() if v["alive_cells"] > 0)
    for ts in timestep_data["nca_stats"]
]

bits_encoded = [
    np.mean([v["shannon_entropy"] for v in ts.values() if v["alive_cells"] > 0])
    for ts in timestep_data["nca_stats"]
]

# Group bits_encoded by n_alive
grouped_data = defaultdict(list)
for n, bits in zip(n_alive, bits_encoded):
    grouped_data[n].append(bits)

# Calculate mean and std for each group
n_alive_unique = sorted(grouped_data.keys())
means = [np.mean(grouped_data[n]) for n in n_alive_unique]
stds = [np.std(grouped_data[n]) for n in n_alive_unique]

# Plot with error bars, colored by number of NCAs alive
plt.figure(figsize=(10, 6))

for n, mean, std in zip(n_alive_unique, means, stds):
    # Use the n-1 index color (since 1 NCA alive = index 0, etc.)
    # Handle case where n might be 0 or exceed color list length
    color_idx = n - 1 if n > 0 else 0
    if color_idx < len(nca_colors):
        color = nca_colors[color_idx][:3]  # Get RGB, ignore alpha if present
    else:
        color = plt.cm.tab10(color_idx)  # Fallback for indices beyond nca_colors

    plt.errorbar(
        n,
        mean,
        yerr=std,
        fmt="o",
        capsize=5,
        capthick=2,
        markersize=8,
        color=color,
        ecolor=color,
    )  # Error bar same color

plt.xlabel("NCAs alive")
plt.ylabel("Mean bits encoded/channel")
plt.title("Bits Encoded vs Number of NCAs Alive")
plt.grid(True, alpha=0.3)

plt.savefig(f"{experiment_dir}/n_vs_bits_{steps_taken}", dpi=300)